# Compare agent configurations with durable evaluation

For Python application authors who can already construct a `finstack_ai.Agent`.
This notebook runs offline with a deterministic Python model callback and the
real Rust execution, journal, scoring, reporting and SQLite paths. No credentials
or external services are needed.

We will bind two configurations to the same dataset, inspect paired quality and
usage/cost coverage, then rescore recorded runs without calling the subject model.
One dataset item intentionally fails to show how failed output and unknown cost
remain visible. This is a framework demonstration, not a model-quality benchmark.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

from finstack_ai import Agent, PythonModel
from finstack_ai import eval as ev

storage = TemporaryDirectory(prefix="finstack-eval-")
root = Path(storage.name)
calls = {"baseline": 0, "candidate": 0}

## Configure the same model with two instructions

The deterministic callback recognizes the configuration instruction in the
actual normalized model request. Both configurations use the same component
identity and pricing policy. SQLite journals are separate from the evaluation
store. `document_tools=False` leaves this fixture with no callable tools.

In [ ]:
async def offline_model(context, request):
    encoded = json.dumps(request)
    candidate = "Use the policy lookup" in encoded
    arm = "candidate" if candidate else "baseline"
    calls[arm] += 1
    if "unavailable fixture" in encoded:
        raise RuntimeError("intentional unavailable fixture")
    answer = "unknown"
    if candidate:
        answer = "seven years" if "retention" in encoded else "24 hours"
    tokens, cost = (8, 7) if candidate else (10, 10)
    return {
        "text": answer,
        "completion_id": f"{arm}-{calls[arm]}",
        "usage": {
            "input_tokens": tokens,
            "output_tokens": 4,
            "total_tokens": tokens + 4,
            "cost": {
                "unit": "USD",
                "micros": str(cost),
                "pricing_policy_version": "v1",
            },
        },
    }


model = PythonModel(
    offline_model,
    component="tutorial.model.eval",
    provider="offline",
    model="offline-eval",
    context_window_tokens=1_048_576,
)
pricing = {
    "max_cost": {
        "unit": "USD",
        "micros": "1000000",
        "pricing_policy_version": "v1",
        "unknown_usage": "allow_within_reserved_maximum",
    }
}


async def configured(name, instruction):
    agent = await Agent.from_python(
        model,
        instruction=instruction,
        document_tools=False,
        sqlite_path=str(root / f"{name}-journal.sqlite"),
    )
    return await agent.with_limits(pricing)


baseline = await configured("baseline", "Give a short general answer.")
candidate = await configured("candidate", "Use the policy lookup for an exact answer.")

## Freeze and execute the experiment

Each task × repetition × subject cell owns one actual session. The first subject
is the paired baseline. Subject failure receives a zero exact-match grade; scorer
failure is a separate classification. No finite experiment budget is configured
here, because the intentional failed calls leave cost unknown.

In [ ]:
tasks = [
    ev.TaskSample("retention", "What is the retention policy?", "seven years"),
    ev.TaskSample("response", "What is the response time?", "24 hours"),
    ev.TaskSample("unavailable", "unavailable fixture", "answer"),
]
spec = ev.EvalSpec(
    "offline comparison",
    tasks,
    [ev.SubjectDecl("baseline"), ev.SubjectDecl("candidate")],
    ["exact_match"],
    repetitions=2,
    limits=ev.EvalLimits(max_concurrency=2, max_replacement_attempts=0),
)
store = ev.SqliteEvalStore(root / "experiment.sqlite")
subjects = [
    ev.SubjectBinding("baseline", baseline),
    ev.SubjectBinding("candidate", candidate),
]
runner = ev.EvalRunner(spec, store, subjects, [ev.ExactMatchScorer()])
result = await runner.run()
report = result.report
assert report["counts"]["expected"] == 12
assert report["counts"]["subject_failed"] == 4
assert calls == {"baseline": 6, "candidate": 6}
report["counts"]

## Read quality and paired measurements

Repetitions reduce within tasks before computing mean/stderr across tasks.
Paired deltas compare raw grades at matching task/repetition coordinates. Token
and cost comparisons show their own measured coverage; unknown failed-call cost
is not converted to zero. Report statistics and exact money are decimal strings.

In [ ]:
quality = [
    {
        "subject": item["subject"],
        **item["statistics"],
        "scored_cells": item["scored_cells"],
        "expected_cells": item["expected_cells"],
    }
    for item in report["aggregates"]
]
paired = report["comparisons"][0]
print("Quality:", quality)
print("Paired quality:", paired["metrics"][0])
print("Paired total tokens:", paired["total_token_delta"])
print("Paired costs:", paired["costs"], "missing pairs:", paired["missing_cost_pairs"])
print("Spending and coverage:", report["spending"])
assert paired["missing_cost_pairs"] == 2
assert report["spending"]["subject_unknown"] == 4

## Gate, export and rescore

This gate requires mean quality of at least 0.5 and no paired regression. Its
independent flags distinguish measured quality from infrastructure, scoring and
coverage failures. Exports omit inputs, targets, transcripts and arbitrary scorer
metadata. Explicit rescoring appends a versioned pass, reading subject journals.

In [ ]:
gate = ev.ThresholdGate(
    "candidate",
    {"scorer": "exact_match", "scorer_version": 1, "name": "exact_match"},
    minimum_mean=500_000,
    maximum_regression=0,
)
print("Gate:", result.evaluate(gate))
result.export_jsonl(root / "attempts.jsonl")
result.write_summary(root / "summary.json")
assert len((root / "attempts.jsonl").read_text().splitlines()) == 12
before = calls.copy()
rescored = await ev.EvalRunner(
    spec, store, subjects, [ev.ExactMatchScorer(version=2)]
).rescore()
assert calls == before
assert all(
    len(records[0]["scores"]) == 2 for records in rescored.snapshot["attempts"].values()
)
print("Rescored without subject calls:", calls)
print(
    "Latest scorer versions:",
    sorted({a["metric"]["scorer_version"] for a in rescored.report["aggregates"]}),
)

## Exercise: require perfect quality

Evaluate the latest scorer version with a minimum mean of one. Predict which
failure flag becomes true. The intentional failed subject results are scored
zeros, so complete grading coverage does not imply perfect quality.

To use live providers, replace the model/agent construction, supply explicit
credentials and accepted pricing policy, and keep the experiment and journal
stores. For cancellation, call `runner.cancel()` and await the active operation;
dropping an await alone leaves its Rust owner active. Judge rescoring may spend
on new grader calls, even though it never re-executes subjects.

In [ ]:
strict = ev.ThresholdGate(
    "candidate",
    {"scorer": "exact_match", "scorer_version": 2, "name": "exact_match"},
    minimum_mean=1_000_000,
)
strict_result = rescored.evaluate(strict)
assert strict_result["quality_failed"] and not strict_result["incomplete"]
strict_result